# 02. Baseline

In [10]:
import os, sys, random, warnings, json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score, precision_score, recall_score, roc_auc_score
import joblib

sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
from preprocessing import get_audio_features
from modeling import split_data, evaluate_model, SEED
random.seed(SEED)

## 1. Подгрузка данных

In [11]:
DATA_PROCESSED = '../data/processed/spotify_processed.csv'
MODELS_DIR = '../models'
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs('../report/images', exist_ok=True)

df = pd.read_csv(DATA_PROCESSED)
audio_features = get_audio_features()
X, y = df[audio_features],df['is_popular']
print(f'X: {X.shape}, y: {y.shape}, Class 0: {(y==0).sum()}, Class 1: {(y==1).sum()}')

X: (114000, 9), y: (114000,), Class 0: 109154, Class 1: 4846


## 2. Разбиение

In [12]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)
print(f'Train {X_train.shape[0]} Val {X_val.shape[0]} Test {X_test.shape[0]} (15%)')
split_info = {'train': int(X_train.shape[0]), 'val': int(X_val.shape[0]), 'test': int(X_test.shape[0]), 'seed': SEED}
with open(os.path.join(MODELS_DIR, 'split_info.json'), 'w') as f:
    json.dump(split_info, f)

Train 79800 Val 17100 Test 17100 (15%)


## 3. Нормализация данных

In [13]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, os.path.join(MODELS_DIR, 'scaler.pkl'))

['../models\\scaler.pkl']

Утечка данных (Data Leakage) была предотвращена за счёт разбиние на три независимых выборки (train, val, test) и за счёт нормализации данных и приведения признаков к одному масштабу.

## 4. Определение качества модели

In [14]:
def evaluate_model(y_true, y_pred, name=''):
    metrics = {'F1-Score': f1_score(y_true, y_pred), 'Accuracy': accuracy_score(y_true, y_pred),
               'Precision': precision_score(y_true, y_pred), 'Recall': recall_score(y_true, y_pred),
               'ROC-AUC': roc_auc_score(y_true, y_pred)}
    if name:
        print(f'{name}:')
        for k, v in metrics.items():
            print(f'{k}: {v:.4f}')
    return metrics

F1 выбрана как основная метрика, т.к. у нас есть сильный дисбаланас классов и нам важнен конечный класс, поэтому не roc-auc/pr-auc и т.п.

## 5. Обучение модели

In [15]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train_scaled, y_train)
y_val_pred = baseline.predict(X_val_scaled)
y_test_pred = baseline.predict(X_test_scaled)
val_metrics = evaluate_model(y_val, y_val_pred, 'Val')
test_metrics = evaluate_model(y_test, y_test_pred, 'Test')
joblib.dump(baseline, os.path.join(MODELS_DIR, 'baseline.pkl'))

Val:
F1-Score: 0.0000
Accuracy: 0.9575
Precision: 0.0000
Recall: 0.0000
ROC-AUC: 0.5000
Test:
F1-Score: 0.0000
Accuracy: 0.9575
Precision: 0.0000
Recall: 0.0000
ROC-AUC: 0.5000


['../models\\baseline.pkl']

## 6. Выводы

In [16]:
print(f'F1: Val={val_metrics["F1-Score"]:}, Test={test_metrics["F1-Score"]:}')
print(classification_report(y_test, y_test_pred))

F1: Val=0.0, Test=0.0
              precision    recall  f1-score   support

           0       0.96      1.00      0.98     16373
           1       0.00      0.00      0.00       727

    accuracy                           0.96     17100
   macro avg       0.48      0.50      0.49     17100
weighted avg       0.92      0.96      0.94     17100



Ожидаемо, что из-за отсутсвия линейных зависимостей, высокого дизбаланса классов мы получили F1 = 0. Baseline модель всегда предсказывает 0, т.е. просто говорит что все треки непопулярные. Нужно применять более сложные алгоритмы.